### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1176811d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x117681bd0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### BaseModel
aseModel is a Pydantic parent class.

It helps you create a structured data model with rules.

BaseModel = base class that gives validation + structure to your data

In [3]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field( description="The director of the movie")
    rating: float = Field( description="The rating of the movie out of 10")


In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1176811d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x117681bd0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [9]:
model.invoke("provide de" \
"tails about the movie Inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 sci-fi action film directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. The story revolves around a concept called "inception," which is the act of planting an idea into someone\'s subconscious. \n\nThe protagonist, Dom Cobb, is a professional thief who steals secrets by entering people\'s dreams. The central conflict is that he\'s offered a chance to erase his criminal past by performing the inverse: planting an idea instead. This is the inception part. The plot involves multiple layers of dreams, each with different reality rules, and a team of specialists each with unique skills. There\'s a lot of action, especially the rotating hallway fight scene, which is one of the most memorable parts. The ending is ambiguous, with a spinning top that could either fall (indic

In [10]:
response= model_with_structure.invoke("provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside Structure

parsed = clean structured data that Python can use easily

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")


# Ask model to return output in Movie structure
model_with_structure = model.with_structured_output(Movie, include_raw=True)
 #include_raw=True allows us to see the raw output from the model in addition to the structured output.
response = model_with_structure.invoke(
    "Give me details about the movie Inception"
)

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools available. There\'s a Movie function that requires title, year, director, and rating. I need to find the correct information for Inception. The title is obviously "Inception". The release year was 2010. The director is Christopher Nolan. As for the rating, it\'s popular, so maybe around 8.8 on IMDb. I should structure the tool call with these parameters. Let me confirm the details to make sure. Yeah, that\'s correct. So the function call should include all four required fields.\n', 'tool_calls': [{'id': '5gaqbfhkp', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 175, 'prompt_tokens': 232, 'total_tokens': 407, 'completion_time': 0.312988246, 'completion_tokens_details': {'reaso

### Nested Structure

In [15]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Mal'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Suzuki')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

A dictionary with fixed keys and fixed value types.

In [16]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_typeddict = model.with_structured_output(MovieDict)

response = model_with_typeddict.invoke(
    "Please provide the details of the movie Avengers"
)

response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [19]:
from typing_extensions import TypedDict, Annotated, NotRequired


class Actor(TypedDict):
    name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field( None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Joker / Bane'}],
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [ ]:
#This model.profile tells you what this LLM model can and cannot do.
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}